# 03 — XLM-R Fine-Tuning (English-Only, Multi-Seed)

Trains XLM-R-base as the encoder baseline across seeds [42, 123, 456] and pushes one repo per seed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/qwen-sentiment'
assert os.path.isdir(PROJECT_ROOT), f'Project not found at {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
%pip install -q -r requirements-colab.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Use T4 runtime.'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import importlib
import src.train_xlmr
importlib.reload(src.train_xlmr)
from src.train_xlmr import train_xlmr

In [ ]:
import os
import wandb
from src.train import load_config
from src.dataset import load_all_splits
from src.train_xlmr import train_xlmr

config = load_config('configs/training_config.yaml')
SEEDS = config.get('training', {}).get('seeds', [42, 123, 456])
print('Seeds:', SEEDS)

In [ ]:
data = load_all_splits(['english'])
train_dataset = data['english']['train']
val_dataset = data['english']['validation']
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

In [ ]:
# Cell 7: Load API keys
import os

def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

# Priority: Colab Secrets > env var > .env file
for key in ['WANDB_API_KEY', 'HF_TOKEN']:
    val = _colab_secret(key) or os.environ.get(key)
    if not val:
        # Try .env file
        try:
            from dotenv import load_dotenv
            load_dotenv('.env')
            val = os.environ.get(key)
        except ImportError:
            pass
    if val:
        os.environ[key] = val
        print(f'  {key}: loaded ✅')
    else:
        print(f'  {key}: NOT FOUND ⚠️ (will use interactive login)')

In [ ]:
# Cell 8: Authenticate W&B
import wandb
wandb.login()

In [ ]:
# Cell 9: Authenticate Hugging Face
import os
if os.environ.get('HF_TOKEN'):
    print('HF_TOKEN detected in environment. Using it automatically.')
else:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
for seed in SEEDS:
    print('\n' + '=' * 80)
    print(f'XLM-R seed {seed}')
    print('=' * 80)

    config['xlmr']['seed'] = int(seed)

    # Fresh W&B run per seed
    try:
        wandb.finish()
    except Exception:
        pass
    wandb.init(
        project=config.get('wandb', {}).get('project', 'qwen-sentiment-crosslingual'),
        entity=config.get('wandb', {}).get('entity'),
        name=f'xlmr-base-s{seed}',
        tags=['xlm-roberta-base', 'encoder-baseline', 'sentiment', 'english-only', f'seed-{seed}'],
    )

    model, tokenizer, trainer = train_xlmr(config, train_dataset, val_dataset, seed=seed)

    repo_id = f'Mrigank005/xlm-roberta-sentiment-crosslingual-seed-{seed}'
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f'✅ Pushed: https://huggingface.co/{repo_id}')

    wandb.finish()